# Voxelize fibril point clouds 

## Imports & define paths & quick functions

In [ ]:
# Imports
import pathlib
import numpy as np
import matplotlib.pyplot as plt
import open3d as o3d
from scipy.spatial.transform import Rotation as R
from tqdm import tqdm
import re

# From script in same folder as notebook!
from custom_fibril_gen.custom_fibril_gen_parallel_v1 import *

In [ ]:
# Define paths
notebookPath = pathlib.Path.cwd()

basePath = pathlib.Path('/Users/andrew/Library/CloudStorage/OneDrive-UCB-O365/research/data_analysis/rsoxs_suite')

savePath = notebookPath.joinpath('open3d_MD_outputs_v1')
fibrilsPath = notebookPath.joinpath('custom_fibril_gen', f'fixed_fibrils')

In [ ]:
# Define simple functions
def to_s0to1(value, min=-180, max=180):
    """Adjust a linear scale from min to max to fit between 0 and 1"""
    shift = 0 - min
    max = max + shift
    return (value + shift) / max

def from_s0to1(value, min=-180, max=180):
    """Inverse of 'to_s0to1': Adjust a linear scale 0 to 1 to an arbitrary linear scale between min and max"""
    shift = 0 + min
    max = max - shift
    return (value * max) + shift

### Load sulfur atomic positions from MD cells

In [ ]:
def strip_numbers(element):
        match = re.match(r"([a-zA-Z]+)", element)
        return match.group(1) if match else element
    
def load_xyz(xyz_path):
    """
    Loads the atomic symbols and coordinates from an XYZ file.

    Parameters:
    - xyz_path: string, path to xyz file of molecule, NP, etc.

    Returns:
    - coords: 2D numpy array of x, y, z coordinates.
    - elements: 1D numpy array of element species for each coord in coords.
    """
    # Extracting the atomic symbols and positions from the XYZ file
    with open(xyz_path, 'r') as file:
        lines = file.readlines()
    symbols = []
    coords = []
    for line in lines[2:]:  # Skipping the first two lines (header and comment)
        parts = line.split()
        if len(parts) < 4:
            # Skip lines that do not have at least 4 parts (symbol + 3 coordinates)
            print(f"Skipping line due to insufficient parts: {line.strip()}")
            continue
        try:
            symbol = strip_numbers(parts[0])
            x, y, z = map(float, parts[1:4])
            symbols.append(symbol)
            coords.append([x, y, z])
        except ValueError as e:
            # Skip lines where conversion to float fails or any other issue arises
            print(f"Skipping line due to error: {line.strip()} ({e})")
            continue
    # Convert lists to numpy arrays
    coords = np.array(coords)
    elements = np.array(symbols)
    
    return coords, elements

def load_pdb(pdb_path):
    """
    Parameters:
    - pdb_path: string, path to pdb file of molecule, protein, etc.

    Returns:
    - coords: 2D numpy array of x, y, z coordinates
    - elements: 1D numpy array of element species for each coord in coords
    """
    coords = []
    elements = []

    # Open and read the PDB file
    with open(pdb_path, 'r') as file:
        for line in file:
            if line.startswith("ATOM") or line.startswith("HETATM"):
                # Extracting the relevant information from ATOM/HETATM lines
                element = line[76:78].strip()  # Element symbol, typically in columns 77-78
                x = float(line[30:38])  # X coordinate, typically in columns 31-38
                y = float(line[38:46])  # Y coordinate, typically in columns 39-46
                z = float(line[46:54])  # Z coordinate, typically in columns 47-54

                elements.append(element)
                coords.append([x, y, z])

    coords = np.array(coords)
    elements = np.array(elements)
    
    return coords, elements

In [ ]:
initialMDPath = basePath.joinpath('local_data/PM6_MD_sim/simulations/initial')
initial_allMDPath = basePath.joinpath('local_data/PM6_MD_sim/simulations/initial_all')
equilibratedMDPath = basePath.joinpath('local_data/PM6_MD_sim/simulations/equilibrated')
decamerMDPath = basePath.joinpath('local_data/PM6_MD_sim/simulations/decamer')
xyzMDPath = basePath.joinpath('local_data/PM6_MD_sim/simulations/PM6_xyz')

newBuildPath = basePath.joinpath('local_data/PM6_MD_sim/simulations/4x16_dimers/build')
newEqPath = basePath.joinpath('local_data/PM6_MD_sim/simulations/4x16_dimers/equilibration')

In [ ]:
list(initial_allMDPath.glob('ortho*pdb'))

## Super cell that saves out voxels for all MD paths in loop:

In [ ]:
# for MD_path in tqdm(sorted(xyzMDPath.glob('*xyz'))[:2:-1]):
# for MD_path in sorted(equilibratedMDPath.glob('*pdb')):
# for MD_path in tqdm(sorted(initial_allMDPath.glob('*pdb'))[:]):
for MD_path in tqdm(sorted(newEqPath.glob('*pdb'))[:]):
    print(MD_path.name)

In [ ]:
vox_size = 0.2  # voxel size in nanometers

# for MD_path in tqdm(sorted(initial_allMDPath.glob('*pdb'))[:]):
# for MD_path in tqdm(sorted(xyzMDPath.glob('*xyz'))[:2:-1]):
# for MD_path in tqdm(sorted(equilibratedMDPath.glob('*pdb'))):

# for MD_path in sorted(newBuildPath.glob('*pdb'))[:]:
for MD_path in sorted(newEqPath.glob('*pdb'))[:]:
    
    if MD_path.suffix=='.pdb':
        coords, elements = load_pdb(str(MD_path))
    elif MD_path.suffix=='.xyz':
        coords, elements = load_xyz(str(MD_path))
    
    MD_name = MD_path.stem
    

    # Need rotate all coordinates to align with CyRSoXS convention:
    # Z is film thickness (direction along beam)
    
    # Data is loaded in as (x,y,z) = (lamella stacking, pipi stacking, backbone stacking)
    # So, rotate +90 degrees about x to make it (x,y,z) = (lamella stacking, backbone stacking, pipi stacking)
    
    coord_rotation = R.from_euler('x', 90, degrees=True)
    coords = coord_rotation.apply(coords)

    # Shift to positive values
    coords[:,0] = coords[:,0] - coords[:,0].min()
    coords[:,1] = coords[:,1] - coords[:,1].min()
    coords[:,2] = coords[:,2] - coords[:,2].min()

    # Convert Å to nm:
    coords[:,0] = coords[:,0] / 10
    coords[:,1] = coords[:,1] / 10
    coords[:,2] = coords[:,2] / 10

    print(f'Loaded & corrected {MD_name} coords, next step will take ~6 minutes for 33K atoms...')
    
    ### For each sulfur atom, identify nearest neighbors (likely 2 carbons in thiophene unit) and extract vector perpendicular to their plane (cross product)
    selected_idxs = np.nonzero(elements=='S')[0]
    # print(selected_idxs.shape)
    # triangle_plane_normals = []

    S_colors = [] # Colors will be: phi, theta, elemental Z (not setting S, just set whole-film values for now later on), converted to scale 0 to 1
    
    for i in range(len(coords[elements=='S'])):    
        distances = np.linalg.norm(coords-coords[elements=='S'][i], axis=1)
        # find 2 shortest distances
        first_neighbor = distances[distances>0].min()
        second_neighbor = distances[distances>first_neighbor].min()
        third_neighbor = distances[distances>second_neighbor].min()
    
        # if first_neighbor < 1.95:
        first_neighbor_idx = np.nonzero(distances==first_neighbor)[0][0]
        selected_idxs = np.append(selected_idxs, first_neighbor_idx)
    
        # if second_neighbor < 1.95:
        second_neighbor_idx = np.nonzero(distances==second_neighbor)[0][0]
        selected_idxs = np.append(selected_idxs, second_neighbor_idx)
    
        A_coord = coords[elements=='S'][i]  # triangle point A (sulfur atom)
        B_coord = coords[first_neighbor_idx]  # triangle point B (first closest carbon)
        C_coord = coords[second_neighbor_idx]  # triangle point C (second closest carbon)
    
        AB = B_coord - A_coord
        AC = C_coord - A_coord
    
        # if np.linalg.norm(AC)<2 and np.linalg.norm(AB)<2:
        plane_vector = np.cross(AB, AC)
        plane_unit_vector = plane_vector/np.linalg.norm(plane_vector)

        # Just set the phi and colors here, no need to do 2 for loops
        phi = np.rad2deg(np.arctan2(plane_unit_vector[0], plane_unit_vector[1]))
        theta = np.rad2deg(np.arccos(plane_unit_vector[2]))   

        S_colors.append([
            to_s0to1(phi, min=-180, max=180), 
            to_s0to1(theta, min=0, max=180), 
            # to_s0to1(S_Z, min=0, max=16)
            1
        ])
    
        # triangle_plane_normals.append(plane_unit_vector)

    # print('Identified and set all sulfur extraordinary vectors')
    
    ### Add all points to their own point clouds
    S_pc = o3d.geometry.PointCloud()
    S_pc.points.extend(coords[elements=='S'])
    S_pc.colors.extend(np.array(S_colors)) 
    
    # Set generic color (angles are irrelevant bc S will be set to 0, volume fraction 1)
    color = [0, 0, 1]
    C_pc = o3d.geometry.PointCloud()
    C_pc.points.extend(coords[elements=='C'])
    C_pc.colors.extend(np.full_like(coords[elements=='C'], color)) 
    
    O_pc = o3d.geometry.PointCloud()
    O_pc.points.extend(coords[elements=='O'])
    O_pc.colors.extend(np.full_like(coords[elements=='O'], color)) 
    
    F_pc = o3d.geometry.PointCloud()
    F_pc.points.extend(coords[elements=='F'])
    F_pc.colors.extend(np.full_like(coords[elements=='F'], color)) 

    # print('Created point clouds')
    
    ### Open3D voxelize!
    S_vg = o3d.geometry.VoxelGrid.create_from_point_cloud(S_pc, voxel_size=vox_size)
    C_vg = o3d.geometry.VoxelGrid.create_from_point_cloud(C_pc, voxel_size=vox_size)
    O_vg = o3d.geometry.VoxelGrid.create_from_point_cloud(O_pc, voxel_size=vox_size)
    F_vg = o3d.geometry.VoxelGrid.create_from_point_cloud(F_pc, voxel_size=vox_size)

    # print('Voxelized')
    
    print(S_vg, C_vg, O_vg, F_vg)

    ### Save voxel grid objects!
    savePath = notebookPath.joinpath('open3d_new-MD_outputs_v1')
    
    o3d.io.write_voxel_grid(str(savePath.joinpath(f'S_vg_vox-{vox_size}_{MD_name}.ply')), S_vg)
    o3d.io.write_voxel_grid(str(savePath.joinpath(f'C_vg_vox-{vox_size}_{MD_name}.ply')), C_vg)
    o3d.io.write_voxel_grid(str(savePath.joinpath(f'O_vg_vox-{vox_size}_{MD_name}.ply')), O_vg)
    o3d.io.write_voxel_grid(str(savePath.joinpath(f'F_vg_vox-{vox_size}_{MD_name}.ply')), F_vg)
    
    print('Saved voxel grids')

In [ ]:
voxels = C_vg.get_voxels()  # returns list of voxels
indices = np.stack(list(vx.grid_index for vx in voxels))

In [ ]:
print(indices[:,0].max())
print(indices[:,1].max())
print(indices[:,2].max())

In [ ]:
# visualize, will need to restart kernel after :O
o3d.visualization.draw_geometries([S_vg])

## Use open3D to build and visualize a point cloud

Do piece-wise point clouds, each point cloud for each different material to load into CyRSoXS\
i.e.: one for sulfur points, one for carbon points, one for oxygen points, one for fluorine points

In [ ]:
newBuildPath = basePath.joinpath('local_data/PM6_MD_sim/simulations/4x16_dimers/build')
newEqPath = basePath.joinpath('local_data/PM6_MD_sim/simulations/4x16_dimers/equilibration')

display([f.name for f in newBuildPath.glob('*pdb')])
display([f.name for f in newEqPath.glob('*pdb')])

In [ ]:
# coords,elements = load_pdb(str(list(initial_allMDPath.glob('ortho*RLX*pdb'))[0]))

# for pdb_path in sorted(initial_allMDPath.glob('*build*pdb'))[-1:]:
for pdb_path in sorted(newBuildPath.glob('*pdb'))[:]:
# for pdb_path in sorted(newEqPath.glob('*pdb'))[:]:

    coords, elements = load_pdb(str(pdb_path))
    name = pdb_path.stem

    # Need rotate all coordinates to align with CyRSoXS convention:
    # Z is film thickness (direction along beam)
    
    # Data is loaded in as (x,y,z) = (lamella stacking, pipi stacking, backbone stacking)
    # So, rotate +90 degrees about x to make it (x,y,z) = (lamella stacking, backbone stacking, pipi stacking)
    
    coord_rotation = R.from_euler('x', 90, degrees=True)
    coords = coord_rotation.apply(coords)

    # Shift to positive values
    coords[:,0] = coords[:,0] - coords[:,0].min()
    coords[:,1] = coords[:,1] - coords[:,1].min()
    coords[:,2] = coords[:,2] - coords[:,2].min()

    # Convert Å to nm:
    coords[:,0] = coords[:,0] / 10
    coords[:,1] = coords[:,1] / 10
    coords[:,2] = coords[:,2] / 10

    
    # coords.shape

    # plt.scatter(coords[elements=='S'][:,0], coords[elements=='S'][:,2])
    plt.scatter(coords[elements=='C'][:,0], coords[elements=='C'][:,1], c='black')
    plt.scatter(coords[elements=='S'][:,0], coords[elements=='S'][:,1], c='yellow')
    plt.scatter(coords[elements=='F'][:,0], coords[elements=='F'][:,1], c='purple')
    plt.scatter(coords[elements=='O'][:,0], coords[elements=='O'][:,1], c='red')

    plt.title(name)
    plt.ylabel('lamella stacking (column 0)')
    plt.xlabel('backbone stacking (column 1)')
    plt.show()
    plt.close('all')

In [ ]:
# For each sulfur atom, identify nearest neighbors (likely 2 carbons in thiophene unit) and extract vector perpendicular to their plane (cross product)

selected_idxs = np.nonzero(elements=='S')[0]
print(selected_idxs.shape)

triangle_plane_normals = []
for i in range(len(coords[elements=='S'])):    
    distances = np.linalg.norm(coords-coords[elements=='S'][i], axis=1)
    # find 2 shortest distances
    first_neighbor = distances[distances>0].min()
    second_neighbor = distances[distances>first_neighbor].min()
    third_neighbor = distances[distances>second_neighbor].min()

    # if first_neighbor < 1.95:
    first_neighbor_idx = np.nonzero(distances==first_neighbor)[0][0]
    selected_idxs = np.append(selected_idxs, first_neighbor_idx)

    # if second_neighbor < 1.95:
    second_neighbor_idx = np.nonzero(distances==second_neighbor)[0][0]
    selected_idxs = np.append(selected_idxs, second_neighbor_idx)

    A_coord = coords[elements=='S'][i]  # triangle point A (sulfur atom)
    B_coord = coords[first_neighbor_idx]  # triangle point B (first closest carbon)
    C_coord = coords[second_neighbor_idx]  # triangle point C (second closest carbon)

    AB = B_coord - A_coord
    AC = C_coord - A_coord

    # if np.linalg.norm(AC)<2 and np.linalg.norm(AB)<2:
    plane_vector = np.cross(AB, AC)
    plane_unit_vector = plane_vector/np.linalg.norm(plane_vector)

    triangle_plane_normals.append(plane_unit_vector)

    # print(i)
    # print(f'Index {first_neighbor_idx}: distance: {first_neighbor}')
    # print(f'Index {second_neighbor_idx}: distance: {second_neighbor}')

print(selected_idxs.shape)
print(len(triangle_plane_normals))

In [ ]:
# plt.scatter(coords[elements=='S'][:,0], coords[elements=='S'][:,2])
plt.scatter(coords[selected_idxs][elements[selected_idxs]=='C'][:,0], coords[selected_idxs][elements[selected_idxs]=='C'][:,1], c='black')
plt.scatter(coords[selected_idxs][elements[selected_idxs]=='S'][:,0], coords[selected_idxs][elements[selected_idxs]=='S'][:,1], c='yellow')
# plt.scatter(coords[elements=='F'][:,0], coords[elements=='F'][:,2], c='purple')
# plt.scatter(coords[elements=='O'][:,0], coords[elements=='O'][:,2], c='red')

plt.title(name)
plt.ylabel('lamella stacking (column 0)')
plt.xlabel('backbone stacking (column 1)')
plt.show()

In [ ]:
S_colors_raw = []  # Colors will be: phi, theta, elemental Z (not setting S, just set whole-film values for now later on)
S_colors = []
S_Z = 16

for vector in triangle_plane_normals:
    phi = np.rad2deg(np.arctan2(vector[0], vector[1]))
    theta = np.rad2deg(np.arccos(vector[2]))   
    S_colors_raw.append([phi, theta, S_Z])
    S_colors.append([
        to_s0to1(phi, min=-180, max=180), 
        to_s0to1(theta, min=0, max=180), 
        # to_s0to1(S_Z, min=0, max=16)
        1
    ])

S_colors

In [ ]:
# for color in S_colors:
#     color_raw = [
#         from_s0to1(color[0], min=-180, max=180), 
#         from_s0to1(color[1], min=0, max=180), 
#         from_s0to1(color[2], min=0, max=16)        
#     ]
#     print(color_raw)

In [ ]:
S_pc = o3d.geometry.PointCloud()
S_pc.points.extend(coords[elements=='S'])
S_pc.colors.extend(np.array(S_colors)) 

# Generic color
color = [0, 0, 1]
C_pc = o3d.geometry.PointCloud()
C_pc.points.extend(coords[elements=='C'])
C_pc.colors.extend(np.full_like(coords[elements=='C'], color)) 

O_pc = o3d.geometry.PointCloud()
O_pc.points.extend(coords[elements=='O'])
O_pc.colors.extend(np.full_like(coords[elements=='O'], color)) 

F_pc = o3d.geometry.PointCloud()
F_pc.points.extend(coords[elements=='F'])
F_pc.colors.extend(np.full_like(coords[elements=='F'], color)) 

## Use open3D to voxelize point cloud

In [ ]:
# np.asarray(pc.points).max() / 1024

In [ ]:
vox_size = 0.2  # voxel size in angstroms
S_vg = o3d.geometry.VoxelGrid.create_from_point_cloud(S_pc, voxel_size=vox_size)
C_vg = o3d.geometry.VoxelGrid.create_from_point_cloud(C_pc, voxel_size=vox_size)
O_vg = o3d.geometry.VoxelGrid.create_from_point_cloud(O_pc, voxel_size=vox_size)
F_vg = o3d.geometry.VoxelGrid.create_from_point_cloud(F_pc, voxel_size=vox_size)

print(S_vg, C_vg, O_vg, F_vg)

In [ ]:
# visualize, will need to restart kernel after :O
# o3d.visualization.draw_geometries([S_vg])
o3d.visualization.draw_geometries([C_vg])

### Save open3d voxel grid as binary .ply file

In [ ]:
savePath

In [ ]:
o3d.io.write_voxel_grid(str(savePath.joinpath(f'S_vg_vox{vox_size}.ply')), S_vg)
o3d.io.write_voxel_grid(str(savePath.joinpath(f'C_vg_vox{vox_size}.ply')), C_vg)
o3d.io.write_voxel_grid(str(savePath.joinpath(f'O_vg_vox{vox_size}.ply')), O_vg)
o3d.io.write_voxel_grid(str(savePath.joinpath(f'F_vg_vox{vox_size}.ply')), F_vg)

In [ ]:
# Load saved file & check contents

loaded_voxel_grid = o3d.io.read_voxel_grid(str(savePath.joinpath(f'S_vg_vox{vox_size}.ply')))
loaded_voxel_grid

In [ ]:
# Convert voxel grid to list of voxels, with grid index & color

# voxels = loaded_voxel_grid.get_voxels()  # returns list of voxels
voxels = S_vg.get_voxels()  # returns list of voxels
# indices = np.stack(list(vx.grid_index for vx in voxels))
# colors = np.stack(list(vx.color for vx in voxels))

voxels

In [ ]:
o3d.visualization.EV.set(S_pc)

In [ ]:
voxels = C_vg.get_voxels()  # returns list of voxels
indices = np.stack(list(vx.grid_index for vx in voxels))

In [ ]:
print(indices[:,0].max())
print(indices[:,1].max())
print(indices[:,2].max())

In [ ]:
notebookPath

In [ ]:
savePath

In [ ]:
coords[elements=='S'].shape

In [ ]:
    ### For each sulfur atom, identify nearest neighbors (likely 2 carbons in thiophene unit) and extract vector perpendicular to their plane (cross product)
    selected_idxs = np.nonzero(elements=='S')[0]
    # print(selected_idxs.shape)
    triangle_plane_normals = []
    for i in tqdm(range(len(coords[elements=='S']))):    
        distances = np.linalg.norm(coords-coords[elements=='S'][i], axis=1)
        # find 2 shortest distances
        first_neighbor = distances[distances>0].min()
        second_neighbor = distances[distances>first_neighbor].min()
        third_neighbor = distances[distances>second_neighbor].min()
    
        # if first_neighbor < 1.95:
        first_neighbor_idx = np.nonzero(distances==first_neighbor)[0][0]
        selected_idxs = np.append(selected_idxs, first_neighbor_idx)
    
        # if second_neighbor < 1.95:
        second_neighbor_idx = np.nonzero(distances==second_neighbor)[0][0]
        selected_idxs = np.append(selected_idxs, second_neighbor_idx)
    
        A_coord = coords[elements=='S'][i]  # triangle point A (sulfur atom)
        B_coord = coords[first_neighbor_idx]  # triangle point B (first closest carbon)
        C_coord = coords[second_neighbor_idx]  # triangle point C (second closest carbon)
    
        AB = B_coord - A_coord
        AC = C_coord - A_coord
    
        # if np.linalg.norm(AC)<2 and np.linalg.norm(AB)<2:
        plane_vector = np.cross(AB, AC)
        plane_unit_vector = plane_vector/np.linalg.norm(plane_vector)
    
        triangle_plane_normals.append(plane_unit_vector)

### Build full point cloud with a single selected fibril

In [ ]:
# # Add coordinates for each box, add to point cloud mesh immediately
# pc = o3d.geometry.PointCloud()

# # # Load RBD coords & XYZ euler rotations, specify how many boxes to fill
# # RBD_XYZrots = np.loadtxt(RBDs[0])
# # RBD_sizes = np.round(np.loadtxt(RBDs[1]), 2)
# # RBD_coords = np.loadtxt(RBDs[2]) * 10000  # convert um to Å

# boxes_to_fill = 1  # -1 is all films
# for RBD_XYZrot, RBD_coord, RBD_size in tqdm(zip(RBD_XYZrots[:boxes_to_fill], RBD_coords[:boxes_to_fill], RBD_sizes[:boxes_to_fill]), total=len(RBD_XYZrots[:boxes_to_fill])):
#     # Load correct fibril:
#     if RBD_size[0]==0.02:
#         backbone_axs, backbone_coords, backbone_css, fibril_points_arr = load_fibril(selected_fibrils[0], skip_step=1)
#         print('20nm')
#     elif RBD_size[0]==0.03:
#         backbone_axs, backbone_coords, backbone_css, fibril_points_arr = load_fibril(selected_fibrils[0], skip_step=1)
#         print('30nm')
#     elif RBD_size[0]==0.04:
#         backbone_axs, backbone_coords, backbone_css, fibril_points_arr = load_fibril(selected_fibrils[0], skip_step=1)
#         print('40nm')
#     else:
#         print('no matchin fibrils to load')
#         break
    
#     # Apply euler rotation to each fibril and backbone axial pointers
#     # Apply pre rotation correction if necessary
#     r_pre_rotation = R.from_euler('y', 0, degrees=True)
#     pre_rotated_fibril = r_pre_rotation.apply(fibril_points_arr)
#     pre_rotated_bb_axs = r_pre_rotation.apply(backbone_axs)
#     pre_rotated_bb_coords = r_pre_rotation.apply(backbone_coords)
#     # Rotate corrected fibril orientation
#     # r_RBD = R.from_euler("xyz", RBD_XYZrot, degrees=True)
#     r_RBD = R.from_euler("xyz", [0,0,0], degrees=True)

#     rotated_fibril = r_RBD.apply(pre_rotated_fibril)
#     rotated_backbone_axs = r_RBD.apply(pre_rotated_bb_axs)
#     rotated_backbone_coords = r_RBD.apply(pre_rotated_bb_coords)
    
#     # # Convert rotated backbone points to 2 Euler angles; PARALLEL to backbone (select backbone axial pointer)
#     # phi = np.rad2deg(np.arctan2(rotated_backbone_axs[:,0], rotated_backbone_axs[:,1]))
#     # theta = np.rad2deg(np.arccos(rotated_backbone_axs[:,2]))
#     # euler_ZY_rot = np.vstack((phi,theta)).T
#     # # Assign euler angle to each fibril point by its nearest backbone point:
#     # fibril_ZYrots = np.empty((0,2))
#     # for fibril_point in fibril_points_arr[:]:
#     #     backbone_displacements = backbone_coords - fibril_point
#     #     backbone_distances = np.sqrt(backbone_displacements[:,0]**2 + backbone_displacements[:,1]**2 + backbone_displacements[:,2]**2)
#     #     backbone_minimum_index = (backbone_distances == backbone_distances.min()).nonzero()[0][0]
    
#     #     fibril_ZYrots = np.append(fibril_ZYrots, euler_ZY_rot[backbone_minimum_index].reshape(1,2), axis=0)
        
#     # Convert rotated backbone points to 2 Euler angles; PERPENDICULAR to backbone (extract euler angle for shortest pointer to backbone)
#     fibril_ZYrots = np.empty((0,2))
#     for fibril_point, rot_fibril_point in zip(fibril_points_arr, rotated_fibril):
        
#         backbone_z_displacements = rotated_backbone_coords[:,2] - fibril_point[2]
#         backbone_minimum_index = (backbone_z_displacements == backbone_z_displacements.min()).nonzero()[0][0]
#     # for fibril_point in rotated_fibril[:]:
#     #     backbone_displacements = rotated_backbone_coords - fibril_point
#     #     backbone_distances = np.sqrt(backbone_displacements[:,0]**2 + backbone_displacements[:,1]**2 + backbone_displacements[:,2]**2)
#     #     backbone_minimum_index = (backbone_distances == backbone_distances.min()).nonzero()[0][0]
        
#         backbone_displacements = rotated_backbone_coords - rot_fibril_point
#         shortest_displacement = backbone_displacements[backbone_minimum_index]
#         mag = np.sqrt(shortest_displacement[0]**2+shortest_displacement[1]**2+shortest_displacement[2]**2)
#         shortest_displacement = shortest_displacement / mag  # make a unit vector
#         print(shortest_displacement)
#         phi = np.rad2deg(np.arctan2(shortest_displacement[0], shortest_displacement[1]))
#         theta = np.rad2deg(np.arccos(shortest_displacement[2]))
#         euler_ZY_rot = np.vstack((phi,theta)).T
    
#         fibril_ZYrots = np.append(fibril_ZYrots, euler_ZY_rot, axis=0)
    
#     # Encode Euler angles into RGB values (set B=1, represent S)`
#     fibril_RGB_values = np.array([to_s0to1(fibril_ZYrots[:,0], min=-180, max=180),  # phi
#                                   to_s0to1(fibril_ZYrots[:,1], min=0, max=180),  # theta
#                                   np.ones((fibril_ZYrots[:,0].shape))]).T  # S (ones)
    
#     # Move fibril coordinates to RBD location
#     # moved_fibril = (rotated_fibril + RBD_coord) / 2  # divide by 2 just for now to undo scaling from before
#     moved_fibril = (rotated_fibril + RBD_coord)  
#     # moved_fibril = (fibril_points_arr + RBD_coord)  
    
#     # Add points and colors to point cloud
#     pc.points.extend(moved_fibril)
#     pc.colors.extend(fibril_RGB_values)

In [ ]:
# o3d.visualization.draw_geometries([pc])

In [ ]:
# o3d.io.write_point_cloud('testing.ply', pc)

## Use open3D to voxelize point cloud

In [ ]:
# np.asarray(pc.points).max() / 1024

In [ ]:
1024*1024*128

In [ ]:
voxel_grid = o3d.geometry.VoxelGrid.create_from_point_cloud(pc, voxel_size=10)
voxel_grid

In [ ]:
# visualize, will need to restart kernel after :O
o3d.visualization.draw_geometries([C_vg])

### Save open3d voxel grid as binary .ply file

In [ ]:
savePath

In [ ]:
# scenarios: 
# - a: extraordinary parallel to fibril long axis
# - b: extraordinary perpendicular, radially pointing to long fibril long axis
# - c: extraordinary perpendicular, radial distribution around long fibril long axis
# - d: extraordinary perpendicular, always full in-plane (edge on for π* manifold) 
# - e: extraordinary perpendicular, always out-of-plane (as much as possible given fibril long axis)
# - f: extraordinary askew, totally random orientations 
# - g: extraordinary askew, always out-of-plane (irrespective of fibril long axis)
# - h: extraordinary askew, distribution (irrespective of fibril long axis)

In [ ]:
o3d.io.write_voxel_grid(str(savePath.joinpath('RBD04-index1_voxelgrid_scenario-a_v1.ply')), voxel_grid)


In [ ]:
# o3d.io.write_voxel_grid(str(savePath.joinpath('RBD01_perp2bb_voxel_grid_v2.ply')), voxel_grid)
# o3d.io.write_voxel_grid(str(savePath.joinpath('RBD03_perp2bb_voxel_grid_v1.ply')), voxel_grid)

# o3d.io.write_voxel_grid(str(savePath.joinpath('RBD04_voxelgrid_perp2bb_fixedfibrils_v1.ply')), voxel_grid)

# o3d.io.write_voxel_grid(str(savePath.joinpath('RBD02_para2bb_voxel_grid_v1.ply')), voxel_grid)
# o3d.io.write_voxel_grid(str(savePath.joinpath('RBD02_perp2bb_voxel_grid_v1.ply')), voxel_grid)

In [ ]:
# Load saved file & check contents

loaded_voxel_grid = o3d.io.read_voxel_grid(str(savePath.joinpath('RBD03_perp2bb_voxel_grid_v1.ply')))
loaded_voxel_grid

In [ ]:
# Load saved file & check contents

loaded_voxel_grid = o3d.io.read_voxel_grid(str(savePath.joinpath('RBD03_para2bb_voxel_grid_v1.ply')))
loaded_voxel_grid

In [ ]:
# Convert voxel grid to list of voxels, with grid index & color

voxels = voxel_grid.get_voxels()  # returns list of voxels
# indices = np.stack(list(vx.grid_index for vx in voxels))
# colors = np.stack(list(vx.color for vx in voxels))

voxels